# LPGNet variants A/B — fully nested participant-level evaluation

This notebook evaluates the fold-local LPGNet baseline, Variant A (squeeze–excitation +
attention pooling), and Variant B (control-trained GRU next-step predictor residual).
**Every outer test fold is untouched by feature fitting, hyperparameter search, early
stopping, and epoch selection.**

For each variant, training seed, and outer fold:

1. Build participant-disjoint inner folds using only outer-training participants.
2. Evaluate every candidate on the **same inner folds**, with fold-local LP/GRU fitting.
3. Select by pooled inner **participant log loss** (lower is better), not recording accuracy.
4. Re-estimate early-stopping epochs on held-out outer-training participants.
5. Refit both stages on all outer-training participants for the selected epochs.
6. Predict outer-test recordings once and store exact out-of-fold probabilities.

Participant-level OOF metrics are primary: probabilities are averaged across each person's
recordings so every participant has equal weight. Recording metrics remain available for
comparison with LPGNet. Bootstrap intervals are conditional on fitted OOF predictions;
fold SD is not a CI. Multiple training seeds are supported and are never counted as extra
participants. Increase `TRAIN_SEEDS` for the final publication run if compute permits.

The fully nested search is compute intensive: the default performs 12 candidates × 3 inner folds × 10 outer folds for each variant and seed, plus final refits. `RUN_ABLATIONS` is optional and does not affect the main nested results.


In [ ]:
# ============================================================
# Reproducibility and experiment settings
# ============================================================
import os, time, random, json, urllib.request, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import lfilter
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             log_loss, roc_curve)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 1                 # official LPGNet notebook seed
FOLD_SEED = 0            # official StratifiedKFold seed
N_FOLDS = 10
N_INNER = 3
VAL_FRAC = 0.15
LP_ORDER = 11
WIN, STRIDE = 100, 50
BATCH_WIN, BATCH_REC = 128, 64
LR_WIN, LR_REC = 5e-4, 1e-3
LABEL_SMOOTH = 0.1
L2_REG = 1e-3
MAX_EPOCHS_WIN, MAX_EPOCHS_REC = 1000, 200
PATIENCE, RLR_PATIENCE, MIN_LR = 25, 14, 1e-7
N_BOOT = 1000
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# ------------------------ publication protocol knobs ------------------------
TRAIN_SEEDS = [42]          # e.g. [42, 43, 52] for a final robustness analysis
N_TRIALS = 12               # candidates per variant per outer fold per seed
EPOCHS_PRED = 25
SEG = 500                   # GRU predictor segment length at 50 Hz
RUN_ABLATIONS = False       # fixed-architecture raw/LP/GRU x inference-mode ablation
N_BOOT = 1000


In [ ]:
# ============================================================
# Download PhysioNet data and attach verified clinical metadata
# ============================================================
URL = 'https://physionet.org/static/published-projects/gaitpdb/gait-in-parkinsons-disease-1.0.0.zip'
ZIP = 'gaitpdb.zip'; DATA_DIR = 'gaitpdb_data'
ROOT = os.path.join(DATA_DIR, 'gait-in-parkinsons-disease-1.0.0')
if not os.path.isdir(ROOT):
    if not os.path.exists(ZIP):
        print('Downloading PhysioNet gaitpdb (~300 MB, one-off)...')
        urllib.request.urlretrieve(URL, ZIP)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA_DIR)

records = []
for fn in sorted(os.listdir(ROOT)):
    if not fn.endswith('.txt') or fn[2:4] not in ('Co', 'Pt'):
        continue
    base = fn[:-4]
    arr = np.loadtxt(os.path.join(ROOT, fn), delimiter='\t')
    records.append(dict(file=fn, study=base[:2], subject=base.split('_')[0],
                        label=int(base[2:4] == 'Pt'),
                        signal=arr[:, 1:19].astype(np.float32)))
subjects = sorted({r['subject'] for r in records})
subj_label = {s: next(r['label'] for r in records if r['subject'] == s) for s in subjects}
by_file = {r['file']: i for i, r in enumerate(records)}
by_subject = {s: [i for i, r in enumerate(records) if r['subject'] == s] for s in subjects}
print(f"{len(records)} recordings | {len(subjects)} participants with sensor files | "
      f"{sum(subj_label.values())} PD / {len(subjects)-sum(subj_label.values())} controls")

# demographics.txt is distributed with gaitpdb. Missing values remain missing.
demo_path = os.path.join(ROOT, 'demographics.txt')
demo = pd.read_csv(demo_path, sep='\t', engine='python', usecols=range(20))
demo = demo.loc[:, ~demo.columns.str.startswith('Unnamed')].copy()
demo = demo.dropna(subset=['ID']).copy()
demo['ID'] = demo['ID'].astype(str)
demo = demo.drop_duplicates('ID').set_index('ID')
clinical = {}
for s in subjects:
    row = demo.loc[s] if s in demo.index else None
    hy = pd.to_numeric(row.get('HoehnYahr'), errors='coerce') if row is not None else np.nan
    # Controls are not assigned HY=0 by inference. Only recorded PD values are used.
    verified_hy = float(hy) if subj_label[s] == 1 and pd.notna(hy) else np.nan
    clinical[s] = dict(
        study=s[:2], diagnosis='PD' if subj_label[s] else 'Control',
        hy=verified_hy, hy_label=(f'{verified_hy:g}' if pd.notna(verified_hy)
                                  else ('Unknown' if subj_label[s] else 'Not applicable')),
        hy_status=('verified' if pd.notna(verified_hy) else
                   ('missing' if subj_label[s] else 'not_applicable')),
        age=pd.to_numeric(row.get('Age'), errors='coerce') if row is not None else np.nan,
        gender=pd.to_numeric(row.get('Gender'), errors='coerce') if row is not None else np.nan)
missing_metadata = [s for s in subjects if s not in demo.index]
print('Participants without a demographics join:', missing_metadata)


In [ ]:
# ============================================================
# Official preprocessing and LP residuals
# ============================================================
def preprocess(x):
    # Causal two-tap moving average, then decimation 100 -> 50 Hz.
    return lfilter([0.5, 0.5], 1, x, axis=0)[::2]

def fit_lp(control_sigs, order=LP_ORDER):
    """One predictor per channel; MATLAB-compatible pooled-control fitting."""
    C, conc_all = np.zeros((18, order)), []
    for c in range(18):
        # generateLPresidual.m starts with zero padding and inserts it between walks.
        pieces = [np.zeros(order, np.float32)]
        pieces += [np.concatenate([s[:, c], np.zeros(order, np.float32)])
                   for s in control_sigs]
        conc = np.concatenate(pieces)
        conc_all.append(conc)
        n = len(conc)
        nfft = 1 << (2*n - 1).bit_length()
        f = np.fft.rfft(conc, nfft)
        r = np.fft.irfft(f*np.conj(f), nfft)[:order+1] / n
        R = np.array([[r[abs(i-j)] for j in range(order)] for i in range(order)])
        C[c] = np.linalg.solve(R, r[1:order+1])
    # MATLAB std uses N-1 normalization. No test recordings enter this calculation.
    stds = np.std(np.stack(conc_all, axis=1), axis=0, ddof=1) + 1e-8
    return C, stds

def apply_lp(x, C, stds):
    e = np.stack([lfilter(np.r_[1.0, -C[c]], 1, x[:, c]) for c in range(18)], axis=1)
    return (e / stds).astype(np.float32)

def lp_features(indices, C, stds):
    return {i: apply_lp(records[i]['proc'], C, stds) for i in indices}

for r in records:
    r['proc'] = preprocess(r['signal'])


In [ ]:
# ============================================================
# Fixed participant-disjoint outer folds and assertions
# ============================================================
def make_folds():
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=FOLD_SEED)
    arr = np.array(subjects); y = np.array([subj_label[s] for s in subjects])
    folds = []
    for tr_s, te_s in skf.split(arr, y):
        tr_subj, te_subj = set(arr[tr_s]), set(arr[te_s])
        assert not tr_subj & te_subj
        tr = [i for i, r in enumerate(records) if r['subject'] in tr_subj]
        te = [i for i, r in enumerate(records) if r['subject'] in te_subj]
        assert not ({records[i]['subject'] for i in tr} & {records[i]['subject'] for i in te})
        folds.append(dict(train=tr, test=te))
    return folds

def split_val(indices, frac=VAL_FRAC, seed=SEED):
    """Split participants, never recordings; output participant sets are disjoint."""
    ss = sorted({records[i]['subject'] for i in indices})
    yy = [subj_label[s] for s in ss]
    if len(set(yy)) > 1:
        splitter = StratifiedShuffleSplit(1, test_size=frac, random_state=seed)
        tr_s, va_s = next(splitter.split(np.zeros(len(ss)), yy))
    else:
        perm = np.random.default_rng(seed).permutation(len(ss))
        n_va = max(1, int(round(frac*len(ss))))
        va_s, tr_s = perm[:n_va], perm[n_va:]
    tr_subj, va_subj = {ss[i] for i in tr_s}, {ss[i] for i in va_s}
    assert not tr_subj & va_subj
    return ([i for i in indices if records[i]['subject'] in tr_subj],
            [i for i in indices if records[i]['subject'] in va_subj])

def labels(indices):
    return np.array([records[i]['label'] for i in indices], dtype=np.float32)

FOLDS = make_folds()
for k, f in enumerate(FOLDS):
    tr_n = len({records[i]['subject'] for i in f['train']})
    te_n = len({records[i]['subject'] for i in f['test']})
    print(f"Fold {k}: {tr_n} train participants / {te_n} test participants | "
          f"{len(f['train'])} train / {len(f['test'])} test recordings")


In [ ]:
# ============================================================
# Released LPGNet architecture and official two-stage training
# ============================================================
def official_sepconv(filters, kernel_size, reg):
    """Use the released call; provide an equivalent modern-Keras fallback."""
    try:
        return layers.SeparableConv1D(filters, kernel_size, activation='linear',
                                      padding='valid', kernel_regularizer=reg)
    except TypeError:
        # Newer Keras exposes separate depthwise/pointwise regularizer arguments.
        return layers.SeparableConv1D(filters, kernel_size, activation='linear',
                                      padding='valid', depthwise_regularizer=reg,
                                      pointwise_regularizer=reg)

def build_lpgnet(T=None, n_ch=18, channels=32, kernels=(7,5,3), spat_drop=0.2,
                 drop=0.25, l2=L2_REG, noise=0.01):
    reg = keras.regularizers.l2(l2)
    if isinstance(channels, int):
        channels = [channels]*(len(kernels)-1) + [2*channels]
    inp = layers.Input((None, n_ch))
    x = layers.SpatialDropout1D(spat_drop)(inp)
    x = layers.GaussianNoise(noise)(x)
    for i, (k, c) in enumerate(zip(kernels, channels)):
        x = official_sepconv(c, k, reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('elu', name='embedding' if i == len(kernels)-1 else None)(x)
        if i < len(kernels)-1:
            x = layers.MaxPool1D(2, 2)(x)
    x = layers.GlobalAvgPool1D(name='pooled')(x)
    x = layers.Dropout(drop)(x)
    out = layers.Dense(1, activation='sigmoid', kernel_regularizer=reg, name='det')(x)
    return keras.Model(inp, out)

def build_recording_model(window_model):
    """Official-equivalent stage 2: frozen pooled backbone, new dropout + sigmoid dense layer."""
    backbone = keras.Model(window_model.input, window_model.get_layer('pooled').output)
    backbone.trainable = False
    inp = layers.Input((None, 18))
    x = backbone(inp)
    x = layers.Dropout(0.25)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inp, out)

def official_windows(indices, feats):
    """Exact endpoint convention from Original.ipynb (exact-ending window excluded)."""
    X, y = [], []
    for i in indices:
        s = feats[i]
        for st in range(0, len(s)-WIN, STRIDE):
            X.append(s[st:st+WIN]); y.append(records[i]['label'])
    return np.asarray(X, np.float32), np.asarray(y, np.float32)

def pad_wrap(indices, feats):
    L = max(len(feats[i]) for i in indices)
    return np.stack([np.pad(feats[i], ((0, L-len(feats[i])), (0,0)), mode='wrap')
                     for i in indices]).astype(np.float32)

def callbacks():
    return [keras.callbacks.EarlyStopping('val_loss', patience=PATIENCE,
                                          restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.25,
                                              patience=RLR_PATIENCE, min_lr=MIN_LR)]

def fit_two_stage(builder, cfg, tr, va, feats, max_win=MAX_EPOCHS_WIN,
                  max_rec=MAX_EPOCHS_REC, verbose=0):
    """Official optimization, with caller-specified participant-disjoint tr/va."""
    model = builder(**cfg) if cfg else builder()
    model.compile(optimizer=keras.optimizers.Adam(LR_WIN, clipvalue=0.5),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=['accuracy'])
    Xtr, ytr = official_windows(tr, feats)
    h1 = model.fit(Xtr, ytr, batch_size=BATCH_WIN, epochs=max_win,
                   validation_data=(pad_wrap(va, feats), labels(va)),
                   callbacks=callbacks(), verbose=verbose)
    full = build_recording_model(model)
    full.compile(optimizer=keras.optimizers.Adam(LR_REC, clipvalue=1),
                 loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                 metrics=['accuracy'])
    h2 = full.fit(pad_wrap(tr, feats), labels(tr), batch_size=BATCH_REC, epochs=max_rec,
                  validation_data=(pad_wrap(va, feats), labels(va)),
                  callbacks=callbacks(), verbose=verbose)
    best = dict(window=int(np.argmin(h1.history['val_loss'])+1),
                recording=int(np.argmin(h2.history['val_loss'])+1))
    return model, full, h1.history, h2.history, best

def refit_two_stage(builder, cfg, train_idx, feats, epochs, verbose=0):
    """After epoch selection, refit on every outer-training participant without test data."""
    model = builder(**cfg) if cfg else builder()
    model.compile(optimizer=keras.optimizers.Adam(LR_WIN, clipvalue=0.5),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=['accuracy'])
    X, y = official_windows(train_idx, feats)
    model.fit(X, y, batch_size=BATCH_WIN, epochs=max(1, epochs['window']), verbose=verbose)
    full = build_recording_model(model)
    full.compile(optimizer=keras.optimizers.Adam(LR_REC, clipvalue=1),
                 loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                 metrics=['accuracy'])
    full.fit(pad_wrap(train_idx, feats), labels(train_idx), batch_size=BATCH_REC,
             epochs=max(1, epochs['recording']), verbose=verbose)
    return model, full

def predict_recordings(full, indices, feats):
    return np.array([float(full.predict(feats[i][None], verbose=0)[0,0]) for i in indices])

def predict_window_average(model, indices, feats):
    out = []
    for i in indices:
        X, _ = official_windows([i], feats)
        out.append(float(model.predict(X, verbose=0).mean()))
    return np.array(out)

_m = build_lpgnet()
print(f'LPGNet: {_m.count_params():,} total parameters; '
      f'{sum(int(w.numpy().size) for w in _m.trainable_weights):,} trainable; '
      f'{_m.count_params()+18*LP_ORDER:,} including LP coefficients')
del _m


In [ ]:
# ============================================================
# OOF metrics, participant aggregation, errors, and clinical subgroups
# ============================================================
def metric_dict(y, p):
    y = np.asarray(y, int); p = np.asarray(p, float); yh = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    return dict(n=len(y), accuracy=accuracy_score(y, yh),
                balanced_accuracy=balanced_accuracy_score(y, yh),
                macro_f1=f1_score(y, yh, average='macro', zero_division=0),
                f1_pd=f1_score(y, yh, zero_division=0),
                sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan,
                roc_auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
                log_loss=log_loss(y, np.clip(p,1e-7,1-1e-7), labels=[0,1]))

def oof_frame(rows):
    df = pd.DataFrame(rows)
    df['prediction'] = (df['probability'] >= 0.5).astype(int)
    df['correct'] = (df['prediction'] == df['label']).astype(int)
    df['confidence'] = np.maximum(df['probability'], 1-df['probability'])
    df['nll'] = -(df['label']*np.log(np.clip(df['probability'],1e-7,1)) +
                  (1-df['label'])*np.log(np.clip(1-df['probability'],1e-7,1)))
    for col in ('study','diagnosis','hy','hy_label','hy_status','age','gender'):
        df[col] = df['subject'].map(lambda s: clinical[s][col])
    return df

def participant_oof(df, group_cols):
    """Equal weight per participant; probability is averaged over that person's walks."""
    keys = group_cols + ['subject']
    agg = dict(label=('label','first'), fold=('fold','first'), probability=('probability','mean'),
               recordings=('file','nunique'))
    out = df.groupby(keys, as_index=False).agg(**agg)
    return oof_frame(out.to_dict('records'))

def summarize_oof(df, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        rec = metric_dict(g['label'], g['probability'])
        part = participant_oof(g, group_cols)
        pm = metric_dict(part['label'], part['probability'])
        rows.append({**base, **{'recording_'+k:v for k,v in rec.items()}})
        rows[-1].update({'participant_'+k:v for k,v in pm.items()})
    return pd.DataFrame(rows)

def wilson_interval(correct, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = correct/n; den = 1+z*z/n
    center = (ph+z*z/(2*n))/den
    half = z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    return center-half, center+half

def error_tables(df, group_cols):
    part = participant_oof(df, group_cols)
    errors = part[part['correct'] == 0].sort_values(group_cols+['nll'], ascending=[True]*len(group_cols)+[False])
    pd_rows = []
    for keys, g in part[part['label'] == 1].groupby(group_cols+['hy_label'], dropna=False):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['hy_label'], keys))
        n = len(g); correct = int(g['correct'].sum()); lo, hi = wilson_interval(correct, n)
        pd_rows.append({**base, 'pd_participants':n, 'correct':correct, 'errors':n-correct,
                        'pd_sensitivity':correct/n, 'wilson_95_low':lo, 'wilson_95_high':hi})
    study_rows = []
    for keys, g in part.groupby(group_cols+['study','diagnosis']):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['study','diagnosis'], keys))
        study_rows.append({**base, **metric_dict(g['label'], g['probability'])})
    return part, errors, pd.DataFrame(pd_rows), pd.DataFrame(study_rows)

def show_table(df, title, limit=100):
    print('\n'+title)
    display(df.head(limit))

def pooled_bootstrap(df, group_cols, metrics=('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity','roc_auc')):
    """Stratified participant bootstrap conditional on fitted OOF predictions."""
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        part = participant_oof(g, group_cols).reset_index(drop=True)
        y = part['label'].to_numpy(); p = part['probability'].to_numpy()
        class_idx = [np.flatnonzero(y == c) for c in (0,1)]
        rng = np.random.default_rng(2026)
        for metric in metrics:
            observed = metric_dict(y,p)[metric]
            vals = []
            for _ in range(N_BOOT):
                ii = np.concatenate([rng.choice(ix, len(ix), replace=True) for ix in class_idx])
                vals.append(metric_dict(y[ii],p[ii])[metric])
            rows.append({**base,'metric':metric,'estimate':observed,
                         'bootstrap_95_low':np.nanpercentile(vals,2.5),
                         'bootstrap_95_high':np.nanpercentile(vals,97.5)})
    return pd.DataFrame(rows)

def plot_confusion_and_roc(df, group_cols, title_col):
    part = participant_oof(df, group_cols)
    groups = list(part.groupby(group_cols))
    fig, axes = plt.subplots(len(groups), 2, figsize=(11,4.3*len(groups)), squeeze=False)
    for row, (keys,g) in enumerate(groups):
        if not isinstance(keys, tuple): keys=(keys,)
        title = ' | '.join(f'{c}={v}' for c,v in zip(group_cols,keys))
        cm = confusion_matrix(g['label'],g['prediction'],labels=[0,1])
        ConfusionMatrixDisplay(cm,display_labels=['Control','PD']).plot(ax=axes[row,0],colorbar=False)
        axes[row,0].set_title(title+' | participant OOF')
        fpr,tpr,_ = roc_curve(g['label'],g['probability'])
        axes[row,1].plot(fpr,tpr,label=f"AUC={roc_auc_score(g['label'],g['probability']):.3f}")
        axes[row,1].plot([0,1],[0,1],'--',color='gray')
        axes[row,1].set(xlabel='False-positive rate',ylabel='True-positive rate',title=title)
        axes[row,1].legend()
    fig.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Variant A and Variant B definitions
# ============================================================
def se_block(x, ratio):
    ch = x.shape[-1]
    s = layers.GlobalAveragePooling1D()(x)
    s = layers.Dense(max(ch//ratio,4),activation='relu')(s)
    s = layers.Dense(ch,activation='sigmoid')(s)
    return layers.Multiply()([x,layers.Reshape((1,ch))(s)])

def attn_pool(x):
    w = layers.Softmax(axis=1)(layers.Dense(1)(x))
    return layers.Lambda(lambda t: tf.reduce_sum(t[0]*t[1],axis=1),name='pooled')([x,w])

def build_variantA(n_ch=18, channels=32, kernels=(7,5,3), se_ratio=8,
                   spat_drop=0.2, drop=0.25, l2=L2_REG, noise=0.01):
    reg=keras.regularizers.l2(l2)
    if isinstance(channels,int): channels=[channels]*(len(kernels)-1)+[2*channels]
    inp=layers.Input((None,n_ch)); x=layers.SpatialDropout1D(spat_drop)(inp)
    x=layers.GaussianNoise(noise)(x)
    for i,(k,c) in enumerate(zip(kernels,channels)):
        x=official_sepconv(c,k,reg)(x); x=layers.BatchNormalization()(x)
        x=layers.Activation('elu')(x); x=se_block(x,se_ratio)
        if i<len(kernels)-1: x=layers.MaxPool1D(2,2)(x)
    x=attn_pool(x); x=layers.Dropout(drop)(x)
    out=layers.Dense(1,activation='sigmoid',kernel_regularizer=reg)(x)
    return keras.Model(inp,out)

BUILDERS={'baseline':build_lpgnet,'A':build_variantA,'B':build_lpgnet}
SPACE_A=dict(channels=[32,48],kernels=[(7,5,3),(9,7,5)],se_ratio=[4,8],
             spat_drop=[0.1,0.2,0.3],drop=[0.25,0.4],l2=[1e-4,1e-3])
SPACE_B=dict(gru_hidden=[16,32],channels=[32,48],spat_drop=[0.1,0.2],
             drop=[0.25,0.4],l2=[1e-4,1e-3])

def classifier_cfg(variant,cfg):
    return {k:v for k,v in cfg.items() if k!='gru_hidden'}

def build_predictor(n_ch=18,hidden=32):
    inp=layers.Input((None,n_ch)); x=layers.GRU(hidden,return_sequences=True)(inp)
    out=layers.TimeDistributed(layers.Dense(n_ch))(x)
    return keras.Model(inp,out)

def pred_segments(indices,mu,sd):
    Xs,Ys=[],[]
    for i in indices:
        z=((records[i]['proc']-mu)/sd).astype(np.float32)
        for st in range(0,len(z)-SEG,SEG):
            seg=z[st:st+SEG]; Xs.append(seg[:-1]); Ys.append(seg[1:])
    return np.stack(Xs),np.stack(Ys)

def train_predictor(train_idx,hidden=32,seed=SEED,verbose=0):
    tf.keras.utils.set_random_seed(seed)
    ctr=[i for i in train_idx if records[i]['label']==0]
    tr,va=split_val(ctr,frac=0.15,seed=seed)
    cat=np.concatenate([records[i]['proc'] for i in tr]); mu,sd=cat.mean(0),cat.std(0)+1e-8
    Xtr,Ytr=pred_segments(tr,mu,sd); Xva,Yva=pred_segments(va,mu,sd)
    model=build_predictor(hidden=hidden)
    model.compile(optimizer=keras.optimizers.Adam(1e-3,clipnorm=1.0),loss='mse')
    model.fit(Xtr,Ytr,validation_data=(Xva,Yva),batch_size=32,epochs=EPOCHS_PRED,
              callbacks=[keras.callbacks.EarlyStopping('val_loss',patience=4,
                                                       restore_best_weights=True)],verbose=verbose)
    return model,mu,sd

def predictor_residual(model,x,mu,sd,rstd):
    z=((x-mu)/sd).astype(np.float32); pred=model.predict(z[None],verbose=0)[0]
    r=np.zeros_like(z); r[1:]=z[1:]-pred[:-1]
    return (r/rstd).astype(np.float32)

def control_resid_std(pred,indices,mu,sd):
    rs=[]
    for i in indices:
        if records[i]['label']!=0: continue
        z=((records[i]['proc']-mu)/sd).astype(np.float32)
        p=pred.predict(z[None],verbose=0)[0]
        r=np.zeros_like(z); r[1:]=z[1:]-p[:-1]; rs.append(r)
    return np.std(np.concatenate(rs),axis=0)+1e-8

def prepare_fold(train_idx,eval_idx,variant,cfg,seed=SEED):
    """Fit on train participants only and transform train+eval; eval never enters fitting."""
    if variant in ('baseline','A'):
        C,stds=fit_lp([records[i]['proc'] for i in train_idx if records[i]['label']==0])
        return lp_features(train_idx+eval_idx,C,stds)
    if variant=='B':
        pred,mu,sd=train_predictor(train_idx,hidden=cfg['gru_hidden'],seed=seed)
        rstd=control_resid_std(pred,train_idx,mu,sd)
        return {i:predictor_residual(pred,records[i]['proc'],mu,sd,rstd)
                for i in train_idx+eval_idx}
    raise ValueError(variant)

def parameter_report(variant,cfg):
    classifier=BUILDERS[variant](**classifier_cfg(variant,cfg))
    n_classifier=int(classifier.count_params())
    n_predictor=18*LP_ORDER if variant in ('baseline','A') else int(build_predictor(hidden=cfg['gru_hidden']).count_params())
    keras.backend.clear_session()
    return n_classifier,n_predictor,n_classifier+n_predictor


In [ ]:
# ============================================================
# Fully nested search: no outer-test participant enters any decision
# ============================================================
def sample_cfg(space,rng):
    return {k:v[int(rng.integers(len(v)))] for k,v in space.items()}

def inner_splits(outer_train,seed):
    ss=sorted({records[i]['subject'] for i in outer_train})
    yy=[subj_label[s] for s in ss]
    skf=StratifiedKFold(N_INNER,shuffle=True,random_state=seed)
    splits=[]
    for tr_s,va_s in skf.split(np.zeros(len(ss)),yy):
        tr_subj={ss[i] for i in tr_s}; va_subj={ss[i] for i in va_s}
        assert not tr_subj & va_subj
        assert va_subj <= {records[i]['subject'] for i in outer_train}
        tr=[i for i in outer_train if records[i]['subject'] in tr_subj]
        va=[i for i in outer_train if records[i]['subject'] in va_subj]
        splits.append((tr,va))
    return splits

def prediction_rows(indices,p,**base):
    return [dict(**base,file=records[i]['file'],subject=records[i]['subject'],
                  label=records[i]['label'],probability=float(prob)) for i,prob in zip(indices,p)]

def evaluate_inner_candidate(variant,cfg,splits,seed,outer_fold,trial,feature_cache):
    rows=[]; epoch_rows=[]; start=time.time()
    for j,(tr,va) in enumerate(splits):
        cache_key=('B',cfg['gru_hidden'],j) if variant=='B' else ('LP',j)
        if cache_key not in feature_cache:
            feature_cache[cache_key]=prepare_fold(tr,va,variant,cfg,seed=seed+100*j)
        feats=feature_cache[cache_key]
        core,early=split_val(tr,seed=seed+1000+j)
        keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed+10000+j)
        model,full,h1,h2,best=fit_two_stage(BUILDERS[variant],classifier_cfg(variant,cfg),
                                            core,early,feats)
        p=predict_recordings(full,va,feats)
        rows += prediction_rows(va,p,family=variant,seed=seed,outer_fold=outer_fold,
                                trial=trial,inner_fold=j,fold=j)
        epoch_rows.append(best)
    inner=oof_frame(rows)
    part=participant_oof(inner,['family','seed','outer_fold','trial'])
    metrics=metric_dict(part['label'],part['probability'])
    return metrics,inner,epoch_rows,time.time()-start

def nested_search(variant,space,outer_fold,outer_train,seed):
    rng=np.random.default_rng(seed+50000*outer_fold+(0 if variant=='A' else 100000))
    splits=inner_splits(outer_train,seed+777*outer_fold)  # identical for every candidate
    trials=[]; selected_inner=[]; best_idx=None; best_key=(np.inf,-np.inf)
    feature_cache={}  # shared by every candidate in this outer search
    for t in range(N_TRIALS):
        cfg=sample_cfg(space,rng)
        metrics,inner,epochs,seconds=evaluate_inner_candidate(variant,cfg,splits,seed,outer_fold,t,feature_cache)
        ncl,npred,ntotal=parameter_report(variant,cfg)
        row=dict(family=variant,seed=seed,outer_fold=outer_fold,trial=t,candidate=cfg,
                 inner_participant_log_loss=metrics['log_loss'],inner_participant_auc=metrics['roc_auc'],
                 inner_participant_accuracy=metrics['accuracy'],
                 mean_best_window_epoch=float(np.mean([e['window'] for e in epochs])),
                 mean_best_recording_epoch=float(np.mean([e['recording'] for e in epochs])),
                 classifier_parameters=ncl,predictor_parameters=npred,total_parameters=ntotal,
                 training_seconds=seconds)
        trials.append(row)
        key=(metrics['log_loss'],-metrics['roc_auc'])
        if key<best_key: best_key=key; best_idx=t; selected_inner=inner
        print(f"[{variant}] outer {outer_fold} seed {seed} trial {t:02d} | "
              f"inner NLL={metrics['log_loss']:.4f} AUC={metrics['roc_auc']:.3f} | {cfg}")
    return best_idx,trials,selected_inner

def fit_selected_outer(variant,cfg,outer_fold,seed):
    f=FOLDS[outer_fold]; tr_all,te=f['train'],f['test']
    feats=prepare_fold(tr_all,te,variant,cfg,seed=seed+200000)
    core,early=split_val(tr_all,seed=seed+300000)
    keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed+400000)
    _,_,h1,h2,best=fit_two_stage(BUILDERS[variant],classifier_cfg(variant,cfg),core,early,feats)
    # Refit on all outer-training participants; test remains untouched.
    keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed+400000)
    model,full=refit_two_stage(BUILDERS[variant],classifier_cfg(variant,cfg),tr_all,feats,best)
    p=predict_recordings(full,te,feats)
    return p,best,(model,full,feats)


In [ ]:
# ============================================================
# Run nested A/B searches and the fixed corrected LPGNet baseline
# ============================================================
all_oof_rows=[]; all_trials=[]; selections=[]; selected_inner_tables=[]
t0=time.time()
for seed in TRAIN_SEEDS:
    for k in range(N_FOLDS):
        # Fixed baseline: no candidate search, but identical outer early stopping/refitting.
        p,best,_=fit_selected_outer('baseline',{},k,seed)
        all_oof_rows += prediction_rows(FOLDS[k]['test'],p,family='baseline',seed=seed,fold=k)
        ncl,npred,ntotal=parameter_report('baseline',{})
        selections.append(dict(family='baseline',seed=seed,outer_fold=k,candidate={},
                               best_window_epoch=best['window'],best_recording_epoch=best['recording'],
                               classifier_parameters=ncl,predictor_parameters=npred,total_parameters=ntotal))
        print(f'[baseline] outer {k} seed {seed} complete')
        for variant,space in [('A',SPACE_A),('B',SPACE_B)]:
            best_idx,trials,inner=nested_search(variant,space,k,FOLDS[k]['train'],seed)
            selected=trials[best_idx]; cfg=selected['candidate']
            all_trials.extend(trials)
            selected_inner_tables.append(inner.assign(selected=True))
            p,best,_=fit_selected_outer(variant,cfg,k,seed)
            all_oof_rows += prediction_rows(FOLDS[k]['test'],p,family=variant,seed=seed,fold=k)
            selections.append(dict(family=variant,seed=seed,outer_fold=k,candidate=cfg,
                                   best_window_epoch=best['window'],best_recording_epoch=best['recording'],
                                   inner_participant_log_loss=selected['inner_participant_log_loss'],
                                   inner_participant_auc=selected['inner_participant_auc'],
                                   classifier_parameters=selected['classifier_parameters'],
                                   predictor_parameters=selected['predictor_parameters'],
                                   total_parameters=selected['total_parameters']))
            print(f'[{variant}] outer {k} seed {seed} SELECTED {cfg}')

nested_oof=oof_frame(all_oof_rows)
trial_table=pd.DataFrame(all_trials)
selection_table=pd.DataFrame(selections)
inner_selected_oof=(pd.concat(selected_inner_tables,ignore_index=True)
                    if selected_inner_tables else pd.DataFrame())
print(f'All nested experiments completed in {(time.time()-t0)/3600:.2f} hours')
show_table(selection_table,'Selected configuration independently inside every outer fold',limit=100)
show_table(trial_table,'All inner-search trials; candidates within each outer fold share inner splits',limit=60)


In [ ]:
# ============================================================
# Complete participant and recording OOF results
# ============================================================
nested_summary=summarize_oof(nested_oof,['family','seed'])
show_table(nested_summary.round(4),'Complete OOF metrics by model family and training seed')
nested_ci=pooled_bootstrap(nested_oof,['family','seed'])
show_table(nested_ci.round(4),'Participant OOF metrics and 95% stratified bootstrap intervals')

across_seed=(nested_summary.groupby('family')[[c for c in nested_summary.columns
             if c.startswith('participant_') and c!='participant_n']].agg(['mean','std']).round(4))
show_table(across_seed,'Training-seed variability on the same outer folds (SD is not participant CI)')

# Fold-level recording metrics are descriptive only.
nested_fold_rows=[]
for (family,seed,fold),g in nested_oof.groupby(['family','seed','fold']):
    nested_fold_rows.append(dict(family=family,seed=seed,fold=fold,
                                 **metric_dict(g['label'],g['probability'])))
nested_fold_metrics=pd.DataFrame(nested_fold_rows)
show_table(nested_fold_metrics.round(4),'Outer-fold recording metrics; training folds overlap, so these are not independent replicates')

plot_confusion_and_roc(nested_oof,['family','seed'],'family')


In [ ]:
# ============================================================
# Paired participant bootstrap: variant minus corrected LPGNet baseline
# ============================================================
def paired_family_bootstrap(df,left,right,metrics=('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity','roc_auc')):
    rows=[]
    seeds=sorted(set(df[df.family==left]['seed']) & set(df[df.family==right]['seed']))
    aligned={}
    for seed in seeds:
        part=participant_oof(df[(df.seed==seed)&(df.family.isin([left,right]))],['family','seed'])
        wide=part.pivot(index='subject',columns='family',values='probability')
        y=part[part.family==left].set_index('subject').loc[wide.index,'label'].to_numpy()
        aligned[seed]=(y,wide[left].to_numpy(),wide[right].to_numpy())
    y=aligned[seeds[0]][0]; classes=[np.flatnonzero(y==c) for c in (0,1)]
    rng=np.random.default_rng(2028)
    for metric in metrics:
        observed=np.mean([metric_dict(y,aligned[s][1])[metric]-metric_dict(y,aligned[s][2])[metric] for s in seeds])
        vals=[]
        for _ in range(N_BOOT):
            ii=np.concatenate([rng.choice(ix,len(ix),replace=True) for ix in classes])
            vals.append(np.mean([metric_dict(aligned[s][0][ii],aligned[s][1][ii])[metric]-
                                 metric_dict(aligned[s][0][ii],aligned[s][2][ii])[metric] for s in seeds]))
        rows.append(dict(comparison=f'{left} minus {right}',metric=metric,
                         seed_averaged_difference=observed,
                         bootstrap_95_low=np.nanpercentile(vals,2.5),
                         bootstrap_95_high=np.nanpercentile(vals,97.5)))
    return pd.DataFrame(rows)

paired_comparisons=pd.concat([paired_family_bootstrap(nested_oof,'A','baseline'),
                              paired_family_bootstrap(nested_oof,'B','baseline')],ignore_index=True)
show_table(paired_comparisons.round(4),
           'Paired variant-minus-baseline differences, averaged over declared seeds; not a prediction ensemble')


In [ ]:
# ============================================================
# Participant errors, HY severity, study patterns, and seed stability
# ============================================================
nested_participants,nested_errors,nested_pd_severity,nested_study=error_tables(nested_oof,['family','seed'])
show_table(nested_errors[['family','seed','fold','subject','study','diagnosis','hy_label','age',
                          'recordings','probability','prediction','confidence','nll']].round(4),
           'All misclassified outer-test participants',limit=250)
show_table(nested_pd_severity.round(4),
           'PD sensitivity by verified HY stage; small cells and Unknown are retained')
show_table(nested_study.round(4),'Participant OOF metrics by study and diagnosis')

seed_stability=(nested_participants.groupby(['family','subject']).agg(
    error_seeds=('correct',lambda x:int((x==0).sum())),n_seeds=('seed','nunique'),
    mean_probability=('probability','mean'),mean_nll=('nll','mean')).reset_index())
seed_stability=seed_stability.merge(nested_participants[['subject','study','diagnosis','hy_label','age']].drop_duplicates('subject'),
                                    on='subject',how='left')
show_table(seed_stability.sort_values(['error_seeds','mean_nll'],ascending=[False,False]).round(4),
           'Persistent versus seed-variable participant errors',limit=200)

if not inner_selected_oof.empty:
    inner_loss=participant_oof(inner_selected_oof,['family','seed','outer_fold','trial'])
    show_table(inner_loss.sort_values('nll',ascending=False)[['family','seed','outer_fold','subject',
               'diagnosis','hy_label','probability','prediction','nll']].head(40).round(4),
               'Highest-NLL selected inner-validation participants; repeated across outer folds, not independent people')


In [ ]:
# ============================================================
# Optional fixed-architecture ablation (no hyperparameter search)
# ============================================================
ablation_oof=pd.DataFrame(); ablation_summary=pd.DataFrame(); ablation_participants=pd.DataFrame()
if RUN_ABLATIONS:
    rows=[]
    for seed in TRAIN_SEEDS:
        for k,f in enumerate(FOLDS):
            tr,te=f['train'],f['test']
            # Raw fold-local normalization uses outer-training controls as the reference cohort.
            cat=np.concatenate([records[i]['proc'] for i in tr if records[i]['label']==0])
            mu,sd=cat.mean(0),cat.std(0)+1e-8
            representations={
                'raw':{i:((records[i]['proc']-mu)/sd).astype(np.float32) for i in tr+te},
                'LP':prepare_fold(tr,te,'baseline',{},seed=seed+600000),
                'GRU':prepare_fold(tr,te,'B',{'gru_hidden':32},seed=seed+600000)}
            for representation,feats in representations.items():
                core,early=split_val(tr,seed=seed+700000)
                keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed+800000)
                _,_,h1,h2,best=fit_two_stage(build_lpgnet,{},core,early,feats)
                keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed+800000)
                model,full=refit_two_stage(build_lpgnet,{},tr,feats,best)
                predictions={'whole_recording':predict_recordings(full,te,feats),
                             'window_average':predict_window_average(model,te,feats)}
                for inference,p in predictions.items():
                    rows += prediction_rows(te,p,representation=representation,inference=inference,
                                            seed=seed,fold=k)
            print(f'[ablation] outer {k} seed {seed} complete')
    ablation_oof=oof_frame(rows)
    ablation_summary=summarize_oof(ablation_oof,['representation','inference','seed'])
    ablation_participants=participant_oof(ablation_oof,['representation','inference','seed'])
    show_table(ablation_summary.round(4),'Fixed-LPGNet representation x inference ablation')
else:
    print('Ablation skipped. Set RUN_ABLATIONS=True to run the fixed-architecture ablation.')


In [ ]:
# ============================================================
# Export complete reproducibility and error-analysis tables
# ============================================================
outdir=Path('lpgnet_nested_outputs'); outdir.mkdir(exist_ok=True)
for name,df in [('recording_oof',nested_oof),('participant_oof',nested_participants),
                ('pooled_metrics',nested_summary),('participant_bootstrap',nested_ci),
                ('fold_metrics',nested_fold_metrics),('paired_comparisons',paired_comparisons),
                ('search_trials',trial_table),('selected_configurations',selection_table),
                ('participant_errors',nested_errors),('pd_severity',nested_pd_severity),
                ('study_metrics',nested_study),('seed_stability',seed_stability),
                ('selected_inner_oof',inner_selected_oof),('ablation_recording_oof',ablation_oof),
                ('ablation_metrics',ablation_summary),('ablation_participant_oof',ablation_participants)]:
    df.to_csv(outdir/f'{name}.csv',index=False)
protocol=dict(outer_folds=N_FOLDS,inner_folds=N_INNER,fold_seed=FOLD_SEED,
              training_seeds=TRAIN_SEEDS,trials_per_variant_per_outer_fold=N_TRIALS,
              selection_metric='pooled inner participant log loss',threshold=0.5,
              final_refit='all outer-training participants using inner-selected configuration and outer-training-selected epochs',
              tensorflow_version=tf.__version__,numpy_version=np.__version__,pandas_version=pd.__version__)
(outdir/'protocol.json').write_text(json.dumps(protocol,indent=2))
print('CSV and protocol outputs written to:',outdir.resolve())


## Reporting checklist and limitations

- Report all outer folds, all declared seeds, every selected configuration, and the actual
  number of participants with sensor files. Do not choose the best seed or best outer fold.
- Participant OOF metrics are primary; recording metrics are secondary. Repeated seeds do
  not create additional participants.
- Bootstrap intervals are conditional on fitted OOF predictions and exclude model-refitting,
  new-split, external-site, and future-population uncertainty.
- The GRU predictor is trained on 500-sample segments but applied continuously to full
  recordings. Treat this as an explicit modeling assumption; a segment-wise inference
  sensitivity analysis is preferable if it materially affects results.
- Variant B's total parameter count includes its GRU predictor. Variant A and B are compact,
  but not necessarily in the original 4,933-parameter regime.
- HY and study analyses are exploratory descriptions, not validated biomarkers or causal
  explanations. `PD sensitivity` means correct detection of PD participants within a stage.
- Any change motivated by these outer errors must be evaluated in a new nested experiment,
  not fed back into the reported folds.
